# Gold-set baseline run (Yasir's adapter) + scoring
Runtime → Change runtime type → **T4 GPU**. Run the cells **in order**. Cells 1–2 are setup, 3 checks the files, 4–5 run the model (about 10–25 min each), 6 prints the score table, 7 downloads the outputs.

In [ ]:
# 1. Clone (or update) the repo and install dependencies
REPO_URL = "https://github.com/m-mirkan/peft-cpe-extraction-pipeline.git"
BRANCH = "feat/codebook-data-curation"

import os
if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt gdown

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 2. Download Yasir's baseline adapter (unchanged, never overwritten)
import os, zipfile
LORA_GDRIVE_ID = "1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX"
LORA_DIR = "checkpoints/qwen25_3b_cpe_lora"
os.makedirs(LORA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(LORA_DIR, "adapter_config.json")):
    !gdown {LORA_GDRIVE_ID} -O lora_adapter.zip
    with zipfile.ZipFile("lora_adapter.zip", "r") as z:
        z.extractall(LORA_DIR)
    entries = os.listdir(LORA_DIR)
    if len(entries) == 1 and os.path.isdir(os.path.join(LORA_DIR, entries[0])):
        nested = os.path.join(LORA_DIR, entries[0])
        for f in os.listdir(nested):
            os.rename(os.path.join(nested, f), os.path.join(LORA_DIR, f))
        os.rmdir(nested)
print("LoRA adapter files:", os.listdir(LORA_DIR))

In [ ]:
# 3. Sanity check: the gold files and the patched script must be there
import json
!ls data/gold scripts
inputs = json.load(open("data/gold/gold_inputs.json", encoding="utf-8"))
gold = json.load(open("data/gold/gold_template.json", encoding="utf-8"))
print("gold titles:", len(inputs), "| unique:", len(set(inputs)), "| labeled:", sum(g["status"] == "labeled" for g in gold))
assert len(inputs) == 100 and [g["input"] for g in gold] == inputs
print("patched inference script (should print 2 lines):")
!grep -n "max_new_tokens=2048\|FAILED" src/infer_and_resolve.py

In [ ]:
# 4. Run titles 1-50 (saves after every title; failures are recorded, not fatal)
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/gold/gold_inputs.json --offset 0 --limit 50 --output_file outputs/gold_base_p1.json

In [ ]:
# 5. Run titles 51-100
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/gold/gold_inputs.json --offset 50 --limit 50 --output_file outputs/gold_base_p2.json

In [ ]:
# 6. Score against the gold set (this is the table to send back)
import json
n1 = len(json.load(open("outputs/gold_base_p1.json", encoding="utf-8")))
n2 = len(json.load(open("outputs/gold_base_p2.json", encoding="utf-8")))
print("predictions saved:", n1, "+", n2, "(expected 50 + 50)")
!python scripts/score_gold.py --gold data/gold/gold_template.json --pred baseline=outputs/gold_base_p1.json,outputs/gold_base_p2.json --json_out outputs/gold_base_scores.json

In [ ]:
# 7. Download the outputs (send me the two prediction files + the table above)
from google.colab import files
files.download("outputs/gold_base_p1.json")
files.download("outputs/gold_base_p2.json")
files.download("outputs/gold_base_scores.json")

### Later (not now): train your own adapter
Only after the codebook is applied to the training data. Use a NEW output dir (e.g. `checkpoints/my_improved_lora`), never the baseline folder. Restart the runtime first to free VRAM.